Step 3: Genetic Algorithm code

In [1]:
#!pip install deap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 656.6 kB/s eta 0:00:00 kB/s eta 0:00:01


In [1]:
import joblib
import pandas as pd
import os

# Load pkl

rf_model_directory = "../ml"
rf_model_filename = "rf_bandgap_dp.pkl"
rf_model_path = os.path.join(rf_model_directory, rf_model_filename)

rf_model = joblib.load(rf_model_path)


In [2]:
# Chemical Space
# A-site (alkaline earth / rare earth)
A_SITES = ["Ba", "Sr", "Ca"]

# B-site transition metals
B_SITES = [
    "Ti", "V", "Cr", "Mn", "Fe",
    "Co", "Ni", "Mo", "W", "Nb", "Ta"
]


In [4]:
# Genome Encoding
import random

def random_individual():
    A = random.randrange(len(A_SITES))
    B, Bp = random.sample(range(len(B_SITES)), 2)
    return [A, B, Bp]


In [5]:
# Build Formula
from pymatgen.core import Composition

def build_formula(individual):
    A = A_SITES[individual[0]]
    B = B_SITES[individual[1]]
    Bp = B_SITES[individual[2]]
    return f"{A}2{B}{Bp}O6"


In [6]:
# Feature Generation
import pandas as pd
from matminer.featurizers.composition import ElementProperty

ep = ElementProperty.from_preset("magpie")

def featurize_formula(formula):
    comp = Composition(formula)
    df = pd.DataFrame({"composition": [comp]})
    X = ep.featurize_dataframe(df, "composition", ignore_errors=True)
    X = X.select_dtypes(include="number")
    return X


/home/mostaq/anaconda3/lib/python3.11/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


In [7]:
# Fitness function
TARGET_BANDGAP = 1.5

def fitness(individual):
    try:
        formula = build_formula(individual)
        X_feat = featurize_formula(formula)
        pred_gap = rf_model.predict(X_feat)[0]

        # fitness = closeness to target
        return (-abs(pred_gap - TARGET_BANDGAP),)

    except Exception:
        return (-10.0,)


In [8]:
#  GA setup
from deap import base, creator, tools

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("individual", tools.initIterate, creator.Individual, random_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

toolbox.register("evaluate", fitness)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutUniformInt,
                 low=[0, 0, 0],
                 up=[len(A_SITES)-1, len(B_SITES)-1, len(B_SITES)-1],
                 indpb=0.2)

toolbox.register("select", tools.selTournament, tournsize=3)


In [ ]:
# GA optimization
POP_SIZE = 80
N_GEN = 100
CX_PB = 0.7
MUT_PB = 0.3

pop = toolbox.population(n=POP_SIZE)

# Initial evaluation
for ind in pop:
    ind.fitness.values = toolbox.evaluate(ind)

for gen in range(N_GEN):
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    # Crossover
    for c1, c2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CX_PB:
            toolbox.mate(c1, c2)
            del c1.fitness.values
            del c2.fitness.values

    # Mutation
    for mutant in offspring:
        if random.random() < MUT_PB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Re-evaluate
    invalid = [ind for ind in offspring if not ind.fitness.valid]
    for ind in invalid:
        ind.fitness.values = toolbox.evaluate(ind)

    pop[:] = offspring

    best = tools.selBest(pop, 1)[0]
    print(f"Gen {gen+1}: Best fitness = {best.fitness.values[0]:.4f}")


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

...

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import joblib

joblib.dump(pop, "ga_final_population.pkl")


In [ ]:
# Top Candidates
top_candidates = tools.selBest(pop, 10)

results = []
for ind in top_candidates:
    formula = build_formula(ind)
    gap = rf_model.predict(featurize_formula(formula))[0]
    results.append((formula, gap))

df_results = pd.DataFrame(results, columns=["formula", "predicted_bandgap"])
df_results
